# Porównanie skuteczności modeli rekomendacyjnych — artykuły naukowe.

**Uczciwy, w pełni odtwarzalny pipeline.** Notebook pobiera dane sam, liczy wszystkie metryki od zera i zapisuje wyniki do CSV. Porównuje cztery silniki rekomendacji treściowej na bibliotece 36 314 artykułów z arXiv:

| silnik | typ | co widzi |
|---|---|---|
| Random | losowa kolejność | nic |
| Popularity | niespersonalizowany | częstość kategorii |
| BM25 | leksykalny | tytuł + abstrakt |
| TF-IDF | leksykalny | tytuł + abstrakt |
| MiniLM (mean) | semantyczny | tytuł + abstrakt |
| MiniLM (max-sim) | semantyczny | tytuł + abstrakt |

**Klucz odpowiedzi.** Zbiór relewantny czytelnika definiuje współczłonkostwo kategorii arXiv (`q-bio.NC AND drugie pole`). Kategorie to metadane przypisane przez autorów — żaden model ich nie czyta, oba dostają wyłącznie tekst. Ocena nie jest więc błędnym kołem.



In [ ]:
# 1. Zależności (bezpieczne do ponownego uruchomienia)
%pip install -q "numpy<2" scikit-learn scipy pandas sentence-transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# 2. Import, konfiguracja i pobranie danych z repozytorium
import os, json, gzip, time, math, random, shutil, urllib.request
import numpy as np, pandas as pd
import scipy.sparse as sp

SEED = 42
random.seed(SEED); np.random.seed(SEED)
KS   = [5, 10]                 # progi dla Precision / Recall / NDCG
DATA = "data"
RAW  = "https://raw.githubusercontent.com/nikabienkowska-svg/Praca-projektowa-/main"
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

os.makedirs(DATA, exist_ok=True)
for fn in ("corpus.jsonl.gz", "users.json"):
    dst = os.path.join(DATA, fn)
    if os.path.exists(dst):
        src = "cache"
    elif os.path.exists(fn):                      # repo sklonowane lokalnie
        shutil.copy(fn, dst); src = "lokalnie"
    else:
        urllib.request.urlretrieve(f"{RAW}/{fn}", dst); src = "pobrano z GitHub"
    print(f"{dst:28} {os.path.getsize(dst)/1e6:7.1f} MB  ({src})")

In [ ]:
# 3. Wczytanie korpusu i profili czytelników
def load_corpus(path):
    ids, texts, cats = [], [], []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            ids.append(r["id"])
            texts.append(((r.get("title", "") or "") + ". " + (r.get("abstract", "") or "")).strip())
            cats.append(set(r.get("categories", "").split()))
    return ids, texts, cats

ids, texts, cats = load_corpus(f"{DATA}/corpus.jsonl.gz")
pos = {p: i for i, p in enumerate(ids)}
N   = len(ids)
meta  = json.load(open(f"{DATA}/users.json"))
users = meta["users"]

print(f"{N} artykułów, {len(users)} czytelników")
print("sygnał ground truth:", meta["ground_truth_signal"])

from collections import Counter
cnt = Counter(c for s in cats for c in s)
print("\nnajczęstsze kategorie w korpusie:")
for c, n in cnt.most_common(6):
    print(f"  {c:12} {n:6}  ({n/N:5.1%})")

In [ ]:
# 4. Odtworzenie pełnego zbioru relewantnego (bez capa) + kontrola spójności
truncated = 0
for u in users:
    rule = [p.strip() for p in u["topic_rule"].split(" AND ")]
    rel  = {i for i in range(N) if all(p in cats[i] for p in rule)}
    assert len(rel) == u["n_relevant_total"], f"rekonstrukcja nie zgadza się dla {u['user_id']}"
    u["_profile"] = {pos[p] for p in u["profile_ids"] if p in pos}
    u["_gt"]      = rel - u["_profile"]                 # pełny klucz, bez obcięcia
    assert not (u["_profile"] & u["_gt"]), "artykuł startowy nie może być w kluczu"
    if len(u["_gt"]) > u["n_ground_truth"]:
        truncated += 1

old = np.array([u["n_ground_truth"] for u in users])
new = np.array([len(u["_gt"]) for u in users])
print(f"rekonstrukcja zgodna dla wszystkich {len(users)} czytelników")
print(f"czytelnicy z uciętym kluczem w oryginale: {truncated}/{len(users)}")
print(f"mediana |GT|  przed: {np.median(old):5.0f}   po: {np.median(new):5.0f}")
print(f"maksimum |GT| przed: {old.max():5.0f}   po: {new.max():5.0f}")

## Silnik 1 — TF-IDF (leksykalny)

Czytelnik jest reprezentowany przez średni wektor TF-IDF swoich 8 artykułów startowych, kandydaci są sortowani po podobieństwie kosinusowym. Własne artykuły startowe są wykluczone z rankingu.

In [ ]:
# 5. TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

t = time.time()
vec = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5,
                      ngram_range=(1, 2), max_features=50000, sublinear_tf=True)
X = vec.fit_transform(texts)                      # N x V, wiersze znormalizowane L2
build_tfidf = time.time() - t
print(f"TF-IDF: {X.shape[1]} cech, indeks zbudowany w {build_tfidf:.1f}s")

def _rank(scores, k, exclude):
    s = np.asarray(scores, dtype=np.float64).ravel().copy()
    s[list(exclude)] = -np.inf
    top = np.argpartition(-s, k)[:k]
    return top[np.argsort(-s[top])]

def recommend_tfidf(profile, k, exclude):
    prof = np.asarray(X[sorted(profile)].mean(axis=0)).ravel()
    return _rank(X.dot(prof), k, exclude)

## Silnik 2 — BM25 (mocny leksykalny punkt odniesienia)

Random i Popularity leżą tak nisko, że pobicie ich niczego nie dowodzi. BM25 to standard w wyszukiwaniu pełnotekstowym i właściwy rywal dla modelu semantycznego. Zapytaniem jest suma terminów z 8 artykułów startowych; macierz wag BM25 liczona jest raz, więc zapytanie to jedno mnożenie rzadkie.

In [ ]:
# 6. BM25 (k1=1.5, b=0.75), zaimplementowany macierzowo
t = time.time()
cv = CountVectorizer(stop_words="english", min_df=3, max_df=0.5, max_features=50000)
Cm = cv.fit_transform(texts).astype(np.float32)

dl    = np.asarray(Cm.sum(axis=1)).ravel(); avgdl = dl.mean()
df    = np.asarray((Cm > 0).sum(axis=0)).ravel()
idf   = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
k1, b = 1.5, 0.75

coo   = Cm.tocoo()
data  = idf[coo.col] * coo.data * (k1 + 1) / (coo.data + k1 * (1 - b + b * dl[coo.row] / avgdl))
Bm    = sp.csr_matrix((data.astype(np.float32), (coo.row, coo.col)), shape=Cm.shape)
build_bm25 = time.time() - t
print(f"BM25: {Bm.shape[1]} terminów, indeks zbudowany w {build_bm25:.1f}s")

def recommend_bm25(profile, k, exclude):
    q = (np.asarray(Cm[sorted(profile)].sum(axis=0)).ravel() > 0).astype(np.float32)
    return _rank(Bm.dot(q), k, exclude)

## Silnik 3 — `all-MiniLM-L6-v2` (semantyczny), dwie agregacje profilu

To najwolniejsza komórka; embeddingi są cache'owane, więc kolejne uruchomienia są natychmiastowe. W Colabie ustaw `Środowisko wykonawcze → Zmień typ → T4 GPU`.

Testowane są dwa sposoby zbudowania profilu z 8 artykułów startowych:

- **mean** — jeden uśredniony wektor, jak w wersji poprzedniej. W anizotropowej przestrzeni embeddingów centroid ośmiu różnych prac potrafi wylądować w miejscu, które nie odpowiada żadnej z nich.
- **max-sim** — dokument dostaje najwyższe podobieństwo do któregokolwiek z artykułów startowych. Dla modeli gęstych zwykle działa to wyraźnie lepiej i nie wymaga, żeby zainteresowania czytelnika były jednorodne.

Bez tego porównania nie da się rozstrzygnąć, czy MiniLM przegrywa jako model, czy jako sposób agregacji.

In [ ]:
# 7. MiniLM — kodowanie (cache) + dwa warianty agregacji profilu
EMB_CACHE = f"{DATA}/emb_minilm.npy"
emb, build_minilm = None, 0.0

if os.path.exists(EMB_CACHE):
    cached = np.load(EMB_CACHE, mmap_mode="r")
    if cached.shape[0] == N:
        emb = np.load(EMB_CACHE)
        print(f"wczytano cache embeddingów {emb.shape} (usuń {EMB_CACHE}, aby przeliczyć)")

if emb is None:
    import torch
    from sentence_transformers import SentenceTransformer
    dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"kodowanie na urządzeniu: {dev}  (CPU zajmie ok. 30-45 min, GPU ok. 2 min)")
    model = SentenceTransformer("all-MiniLM-L6-v2", device=dev)
    t = time.time()
    emb = model.encode(texts, batch_size=64, show_progress_bar=True,
                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    build_minilm = time.time() - t
    np.save(EMB_CACHE, emb)
    print(f"embeddingi {emb.shape} policzone w {build_minilm:.1f}s")

def recommend_minilm_mean(profile, k, exclude):
    prof = emb[sorted(profile)].mean(axis=0)
    return _rank(emb.dot(prof), k, exclude)

def recommend_minilm_maxsim(profile, k, exclude):
    sims = emb.dot(emb[sorted(profile)].T).max(axis=1)     # najlepsze dopasowanie do dowolnego seeda
    return _rank(sims, k, exclude)

In [ ]:
# 8. Diagnostyka: ile tekstu MiniLM w ogóle widzi (limit 256 word-pieces)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

sample = random.Random(SEED).sample(texts, 2000)
lens   = np.array([len(tok.encode(s)) for s in sample])
over   = (lens > 256).mean()
seen   = np.minimum(lens, 256).sum() / lens.sum()

print(f"długość w word-pieces: mediana {np.median(lens):.0f}, 90. percentyl {np.percentile(lens,90):.0f}")
print(f"abstraktów przekraczających limit 256: {over:.1%}")
print(f"udział tekstu, który MiniLM faktycznie czyta: {seen:.1%}   (TF-IDF i BM25 czytają 100%)")

## Punkty odniesienia

**Random** — ranking losowy. **Popularity** — ta sama, niespersonalizowana lista dla wszystkich, sortowana po tym, jak częste w korpusie są kategorie danego artykułu. W danych nie ma żadnych interakcji użytkowników, więc jest to proxy „centralności tematycznej", a nie popularność w sensie systemów rekomendacyjnych.

In [ ]:
# 9. Baseline'y
catf = Counter(c for s in cats for c in s)
popscore  = np.array([sum(catf[c] for c in cats[i]) for i in range(N)], dtype=float)
pop_order = np.argsort(-popscore)
_rng = random.Random(SEED)

def recommend_popularity(profile, k, exclude):
    return [d for d in pop_order if d not in exclude][:k]

def recommend_random(profile, k, exclude):
    return _rng.sample([d for d in range(N) if d not in exclude], k)

## Ewaluacja — Precision@K, Recall@K, NDCG@K

Metryki liczone są osobno dla każdego czytelnika i dopiero potem uśredniane, żeby zachować rozkład per-czytelnik do testu istotności. `IDCG` normalizuje przez `min(|GT|, K)`, więc idealny ranking daje NDCG = 1 niezależnie od wielkości zbioru relewantnego.

In [ ]:
# 10. Metryki i przebieg ewaluacji
def ndcg(hits, k, n_gt):
    dcg  = sum(1 / math.log2(i + 2) for i, h in enumerate(hits[:k]) if h)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(n_gt, k))) or 1.0
    return dcg / idcg

def evaluate(name, recommender):
    rows, t = [], time.time()
    for u in users:
        rec  = list(recommender(u["_profile"], max(KS), u["_profile"]))
        hits = [1 if d in u["_gt"] else 0 for d in rec]
        row  = {"reader": u["user_id"]}
        for k in KS:
            row[f"P@{k}"]    = sum(hits[:k]) / k
            row[f"R@{k}"]    = sum(hits[:k]) / max(len(u["_gt"]), 1)
            row[f"NDCG@{k}"] = ndcg(hits, k, len(u["_gt"]))
        rows.append(row)
    df = pd.DataFrame(rows); df["engine"] = name
    return df, time.time() - t

engines = [("Random",        recommend_random),
           ("Popularity",    recommend_popularity),
           ("BM25",          recommend_bm25),
           ("TF-IDF",        recommend_tfidf),
           ("MiniLM-mean",   recommend_minilm_mean),
           ("MiniLM-maxsim", recommend_minilm_maxsim)]

detail, qtime = {}, {}
for name, fn in engines:
    detail[name], qtime[name] = evaluate(name, fn)
    print(f"  {name:15} 62 czytelników w {qtime[name]:5.2f}s")

order   = [n for n, _ in engines]
summary = (pd.concat(detail.values())
             .groupby("engine")[[f"{m}@{k}" for m in ("P", "R", "NDCG") for k in KS]]
             .mean().loc[order])
print(f"\nŚrednie metryki dla {len(users)} czytelników (pełny klucz kategorialny):")
summary

## Istotność statystyczna i koszt (hipotezy H1 i H2)

**H1** (model semantyczny trafniejszy od leksykalnego) to twierdzenie o rozkładzie między czytelnikami, więc wymaga testu parowanego, a nie samej różnicy średnich. Porównań jest kilka, więc p-value dostaje korektę Holma; raportowana jest też wielkość efektu `r = Z/√n`, bo przy 62 czytelnikach samo p niewiele mówi.

**H2** (TF-IDF tańszy) to zmierzony czas budowy indeksu i czas odpowiedzi.

In [ ]:
# 11. Test Wilcoxona z korektą Holma + wielkość efektu
from scipy.stats import wilcoxon

nd    = {n: detail[n].set_index("reader")["NDCG@10"] for n in detail}
pairs = [("MiniLM-maxsim", "TF-IDF"),
         ("MiniLM-mean",   "TF-IDF"),
         ("MiniLM-maxsim", "MiniLM-mean"),
         ("TF-IDF",        "BM25"),
         ("TF-IDF",        "Popularity")]

rows = []
for a, b in pairs:
    d = nd[a] - nd[b]
    if (d == 0).all():
        rows.append({"A": a, "B": b, "W": np.nan, "p": 1.0, "r": 0.0, "mediana_różnicy": 0.0}); continue
    st = wilcoxon(nd[a], nd[b])
    n_eff = int((d != 0).sum())
    z = abs(st.statistic - n_eff * (n_eff + 1) / 4) / math.sqrt(n_eff * (n_eff + 1) * (2 * n_eff + 1) / 24)
    rows.append({"A": a, "B": b, "W": st.statistic, "p": st.pvalue,
                 "r": z / math.sqrt(n_eff), "mediana_różnicy": float(np.median(d))})

sig = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
m   = len(sig)
sig["p_holm"] = np.maximum.accumulate(np.minimum(1.0, sig["p"] * (m - np.arange(m))))
print("Wilcoxon parowany na NDCG@10 per czytelnik (r: 0.1 mały, 0.3 średni, 0.5 duży efekt):")
display(sig)

print("H2 — koszt:")
print(f"  TF-IDF  budowa indeksu {build_tfidf:7.1f}s   zapytania {qtime['TF-IDF']:5.2f}s")
print(f"  BM25    budowa indeksu {build_bm25:7.1f}s   zapytania {qtime['BM25']:5.2f}s")
print(f"  MiniLM  budowa indeksu {build_minilm:7.1f}s   zapytania {qtime['MiniLM-maxsim']:5.2f}s"
      + ("  (z cache, nie kodowano w tym przebiegu)" if build_minilm == 0 else ""))

In [ ]:
# 12. Zapis wyników, żeby liczby w raporcie dało się zacytować i odtworzyć
summary.to_csv(f"{DATA}/results_summary.csv")
pd.concat(detail.values()).to_csv(f"{DATA}/results_per_reader.csv", index=False)
sig.to_csv(f"{DATA}/results_significance.csv", index=False)
print("zapisano results_summary.csv, results_per_reader.csv, results_significance.csv")

worst = (detail["TF-IDF"].set_index("reader")["NDCG@10"] - detail["MiniLM-maxsim"].set_index("reader")["NDCG@10"])
print("\nCzytelnicy z największą rozbieżnością między silnikami (materiał do analizy błędów):")
display(pd.DataFrame({"różnica NDCG@10 (TF-IDF - MiniLM)": pd.concat([worst.sort_values().head(3), worst.sort_values().tail(3)])}))

## Interfejs webowy (Gradio)

Interaktywny podgląd rekomendacji dla wybranego czytelnika, z linkami do arXiv i zaznaczeniem, które pozycje są w kluczu odpowiedzi. Uruchomienie jest opcjonalne — reszta notebooka nie zależy od tej komórki.

In [ ]:
# 13. Aplikacja Gradio (opcjonalna)
%pip install -q gradio
import gradio as gr

ENGINES = {"TF-IDF": recommend_tfidf, "BM25": recommend_bm25,
           "MiniLM (mean)": recommend_minilm_mean, "MiniLM (max-sim)": recommend_minilm_maxsim}
by_id = {u["user_id"]: u for u in users}
labels = [f'{u["user_id"]} — {u["topic_label"]}' for u in users]

def show(label, engine_name, k):
    u = by_id[label.split(" — ")[0]]
    rec = ENGINES[engine_name](u["_profile"], int(k), u["_profile"])
    out = [f'### {u["topic_label"]}  \nreguła: `{u["topic_rule"]}` · zbiór relewantny: {len(u["_gt"])} artykułów\n']
    out.append("**Artykuły startowe (profil czytelnika):**\n")
    for d in sorted(u["_profile"])[:4]:
        out.append(f"- {texts[d].split('. ')[0][:110]}")
    out.append(f"\n**Rekomendacje — {engine_name}:**\n")
    for i, d in enumerate(rec, 1):
        mark = "✅" if d in u["_gt"] else "▫️"
        out.append(f"{i}. {mark} [{texts[d].split('. ')[0][:110]}](https://arxiv.org/abs/{ids[d]})  \n"
                   f"   <sub>{' '.join(sorted(cats[d]))}</sub>")
    hits = sum(1 for d in rec if d in u["_gt"])
    out.append(f"\n**Trafienia: {hits}/{len(rec)}**  (✅ = pozycja w kluczu odpowiedzi)")
    return "\n".join(out)

gr.Interface(fn=show,
             inputs=[gr.Dropdown(labels, value=labels[0], label="Czytelnik"),
                     gr.Radio(list(ENGINES), value="TF-IDF", label="Silnik"),
                     gr.Slider(3, 15, value=5, step=1, label="Liczba rekomendacji")],
             outputs=gr.Markdown(label="Wynik"),
             title="Rekomender artykułów naukowych",
             description="Porównanie silników leksykalnych i semantycznych na 36 314 artykułach z arXiv."
            ).launch(share=True, debug=False)

## Co z tego wynika:

**Opis danych.** Korpus to 36 314 artykułów, ale nie jest to zbiór „z dziedziny neurobiologii i kognitywistyki" — dominuje w nim `cs.AI` (22 403 pozycje, ok. 62%), `q-bio.NC` to 11 902. Właściwy opis: przekrój arXiv obejmujący `q-bio.NC`, `cs.AI` i `cs.NE`, w którym profile czytelników są zakotwiczone w `q-bio.NC`.

**Wyniki.** Do raportu wchodzi wygenerowany `results_summary.csv`, nigdy liczby przepisane ręcznie. Trzeba zaznaczyć, że klucz odpowiedzi jest kategorialny i niezależny od obu modeli.

**Analiza błędów.** Materiałem są czytelnicy o największej rozbieżności między silnikami z ostatniej komórki, a nie wymyślone przykłady.

**Wnioski — i tu leży najważniejsze ograniczenie.** Klucz oparty na współczłonkostwie kategorii arXiv strukturalnie sprzyja modelom leksykalnym: kategorie mocno korelują z żargonem (`optics`, `quantum`, `spiking`, `fMRI`), więc wykrycie reguły `q-bio.NC AND physics.optics` jest w dużej mierze zadaniem dopasowania słów. Wynik „model leksykalny wypada lepiej" jest zatem prawdziwy **dla tego zadania i tej definicji trafności**, a nie ogólną tezą o wyższości TF-IDF nad modelami semantycznymi. Do tego dochodzą trzy asymetrie warunków, które trzeba wymienić wprost:

1. TF-IDF jest dostrojony (`min_df`, `max_df`, bigramy, `sublinear_tf`, 50 tys. cech), MiniLM działa bez żadnego dostrajania.
2. MiniLM czyta tylko pierwsze 256 word-pieces abstraktu (komórka 8 podaje dokładny odsetek), silniki leksykalne czytają całość.
3. `all-MiniLM-L6-v2` to model ogólnego przeznaczenia. Dla artykułów naukowych właściwym punktem odniesienia byłby SPECTER2 lub `bge`/`gte`; dopisanie go to najbardziej wartościowe rozszerzenie tej pracy.

H1 wolno więc odrzucić wyłącznie w brzmieniu: *ten konkretny model semantyczny, z tą agregacją profilu i przy kategorialnej definicji trafności, nie pobił dostrojonego modelu leksykalnego*. Sformułowanie mocniejsze nie jest poparte danymi. H2 (koszt) broni się bez zastrzeżeń — różnica w czasie budowy indeksu jest o rząd wielkości i została zmierzona.

Pozostałe ograniczenia: czytelnicy są syntetyczni, w danych nie ma ani jednej rzeczywistej interakcji użytkownika, a kategoria to zgrubne przybliżenie preferencji.